# ESMGAT Training Pipeline — Google Colab (Tesla T4 GPU)

This notebook trains the **GAT graph link predictor** (`GATLinkPredictor`) for the TransGraph-PPI project on Google Colab with free NVIDIA T4 GPU acceleration.

### Invariants Preserved:
- Uses the **exact same STRING graph** (19,431 nodes, ~2.2M edges) and 483-dim node features (480 ESM-2 + 3 topological).
- Uses the **exact same train/validation split** (`train.csv`, `val.csv`).
- Uses the **exact same decoder head**, optimizer (`AdamW`), scheduler (`CosineAnnealingLR`), and focal loss as GraphSAGE.
- `test.csv` is **never touched** during training.
- Outputs are saved strictly into `ESMGAT/weights/` and `ESMGAT/checkpoints/`.
- The stable GraphSAGE checkpoints and models are **completely untouched**.

## 1. Verify NVIDIA T4 GPU Allocation

In [ ]:
!nvidia-smi

## 2. Clone Repository and Switch to `gat-exp` Branch

In [ ]:
import os
if not os.path.exists('/content/Majorproject'):
    !git clone https://github.com/AshShenoy14/Majorproject.git /content/Majorproject

%cd /content/Majorproject
!git checkout gat-exp
!git pull origin gat-exp
!pwd

## 2b. Supply the 4 Processed Data Files (~41 MB total)
Because `data/processed/*` is gitignored, supply the 4 required data files:
- `ppi_graph.pt` (~34 MB)
- `ppi_graph_mapping.pt` (~0.35 MB)
- `train.csv` (~5.5 MB)
- `val.csv` (~0.69 MB)

**Method A**: Run the cell below. It will prompt you with a **Choose Files** button to upload the 4 files directly from your computer's `Majorproject/data/processed/` folder.

*(Alternatively, drag-and-drop them into `/content/Majorproject/data/processed/` via the left-side Colab file manager)*.

In [ ]:
import os
from google.colab import files

target_dir = '/content/Majorproject/data/processed'
os.makedirs(target_dir, exist_ok=True)

required = ['ppi_graph.pt', 'ppi_graph_mapping.pt', 'train.csv', 'val.csv']
missing = [f for f in required if not os.path.exists(os.path.join(target_dir, f))]

if missing:
    print(f"Please upload the following missing data files: {missing}")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        dest = os.path.join(target_dir, filename)
        with open(dest, 'wb') as f:
            f.write(content)
        print(f"Saved {filename} to {dest}")
else:
    print("All required data files are already present.")

print("\nConfirmed files in data/processed:")
for fname in sorted(os.listdir(target_dir)):
    fpath = os.path.join(target_dir, fname)
    if os.path.isfile(fpath):
        print(f"  - {fname} ({os.path.getsize(fpath) / 1e6:.2f} MB)")

## 3. Install Dependencies
Install PyTorch Geometric and required packages.

In [ ]:
!pip install torch-geometric==2.7.0 xgboost==3.2.0 scikit-learn==1.7.1

## 4. Run Pre-Flight Sanity & Memory Safety Check
This runs the 11-point verification on the real STRING graph and measures VRAM allocation on the T4 GPU.

In [ ]:
!python ESMGAT/training/train_gat_colab.py --dry_run_only

## 5. Full GAT Training with Early Stopping & Platt Calibration
Runs full training (max 100 epochs, patience 15). Early stopping monitors `val.csv` BCE loss.
Upon completion, Platt calibration is fitted on validation logits and saved.

In [ ]:
!python ESMGAT/training/train_gat_colab.py --epochs 100 --patience 15 --lr 0.001

## 6. Download Artifacts to Local Project
Download the trained GAT weights and calibrator to place into `ESMGAT/weights/` on your local machine.

In [ ]:
from google.colab import files

# Download GAT best model weights (~4.8 MB)
files.download('ESMGAT/weights/gat_model_best.pth')

# Download GAT Platt calibrator (~80 bytes)
files.download('ESMGAT/weights/gat_calibrator.json')

# Download validation metrics and report
files.download('ESMGAT/results/gat_val_metrics.json')
files.download('ESMGAT/results/gat_calibration.json')